# 02. Setting A — 일별 21일 누적 수익률 예측

**Phase 1 — Step 2: LSTM × ETF × 일별(21영업일 뒤) 설정**

## 이 노트북의 역할

| 절 | 내용 | 상태 |
|---|---|---|
| §1 | 환경 설정 (`00_setup_and_utils.ipynb` %run) | ✅ |
| §2 | 데이터 로드 (SPY·QQQ 일별 log-return, 분석 기간 2016–2025) | ✅ |
| §3 | 타깃 생성 (`scripts/targets.py` — 21일 누적 forward log-return) | ✅ |
| §4 | 누수 검증 3종 (assert + 육안 표 + 인공 누수 대조) | ✅ |
| §5 | `LSTMDataset` + `make_sequences` 검증 (seq_len=**126**) | ✅ |
| §6 | Walk-Forward fold 생성 (IS=231, purge=21, emb=21, OOS=21, step=21) | ✅ |
| §7 | LSTMRegressor import — TODO (`scripts/models.py` 필요) | 🔜 |
| §8 | 학습 루프 — TODO (`scripts/train.py` 필요) | 🔜 |
| §9 | 평가·시각화 — TODO (`scripts/metrics.py`, `scripts/plot_utils.py` 필요) | 🔜 |
| §10 | 결론·체크포인트 | 🔜 |

## 설정 A 사양 (PLAN.md 확정값)

| 항목 | 값 |
|------|---|
| 타깃 | 21영업일 후 누적 log-return |
| seq_len | **126** (약 6개월 lookback) |
| hidden_size | 128 |
| num_layers | 2 |
| dropout | 0.2 |
| IS | 231 영업일 (~11개월) |
| OOS | 21 영업일 (~1개월) |

## 누수 방지 설계

- **타깃**: `log_ret.rolling(21).sum().shift(-21)` — 미래 21일 합 (예측 목표, §4 검증)
- **Scaler**: `StandardScaler.fit()` 은 **훈련 구간(train_idx)** 만 사용, val·test는 transform만
- **Walk-Forward**: Purge(21) + Embargo(21) — IS 끝과 OOS 사이 42일 gap 보장
- **테스트 입력 lookback**: purge/embargo 구간과 겹쳐도 **입력 피처**이므로 허용

## 진실원

유틸리티 구현:
- [scripts/targets.py](scripts/targets.py) — 타깃 생성·누수 검증
- [scripts/dataset.py](scripts/dataset.py) — LSTMDataset, make_sequences, walk_forward_folds, build_fold_datasets

## §1. 환경 설정 로드

**무엇을**: `00_setup_and_utils.ipynb` 의 공통 설정을 한 줄로 로드한다.

**왜 이 방식**: 한글 폰트·시드·경로 상수(`RAW_DATA_DIR`, `RESULTS_DIR` 등)를 모든 노트북에서
중복 없이 재사용하기 위해 `%run` 패턴을 사용한다.

**주의**: `%run` 은 노트북의 모든 셀을 순서대로 실행하므로 부작용(side-effect)이 있는 셀이
없어야 한다. `00_setup_and_utils.ipynb` 는 순수 설정 코드만 포함하도록 관리한다.

In [1]:
%run ./00_setup_and_utils.ipynb

[OK] scripts.setup import 완료 — BASE_DIR = /Users/yoonseokim/2025_main_bootcamp/4th_final_project/finance_project/시계열_Test/Phase1_LSTM
[OK] 한글 폰트 설정 완료: AppleGothic
[OK] 시드 고정 완료: SEED=42
[정보] PyTorch 버전: 2.11.0, CUDA 가용: False
[OK] 경로 상수 확인
  BASE_DIR      = /Users/yoonseokim/2025_main_bootcamp/4th_final_project/finance_project/시계열_Test/Phase1_LSTM
  RAW_DATA_DIR  = /Users/yoonseokim/2025_main_bootcamp/4th_final_project/finance_project/시계열_Test/Phase1_LSTM/results/raw_data
  SETTING_A_DIR = /Users/yoonseokim/2025_main_bootcamp/4th_final_project/finance_project/시계열_Test/Phase1_LSTM/results/setting_A
  SETTING_B_DIR = /Users/yoonseokim/2025_main_bootcamp/4th_final_project/finance_project/시계열_Test/Phase1_LSTM/results/setting_B
[OK] 공통 import + 표시 옵션 적용 완료
  pandas 2.3.3, numpy 2.4.4
  Phase 1 — LSTM 단독 베이스라인 / 환경 설정 완료
  한글 폰트  : AppleGothic
  시드       : 42
  결과 경로  : /Users/yoonseokim/2025_main_bootcamp/4th_final_project/finance_project/시계열_Test/Phase1_LSTM/results
  진실원     : scripts/se

## §2. 데이터 로드

**무엇을**: `01_data_download_and_eda.ipynb` 가 저장한 SPY·QQQ CSV를 읽고
분석 기간(2016-01-01 ~ 2025-12-31)으로 절단한다.

**왜 전체 기간에서 먼저 log_return 계산**: 분석 시작일(2016-01-04)의 log_return이
직전 거래일(2015-12-31) 대비로 올바르게 계산되어야 한다.
분석 기간 절단 *후* diff() 를 하면 시작일에 NaN이 생긴다.

**대안과 트레이드오프**: `Adj Close` 대신 `Close` 를 쓰면 배당·액면분할이 반영되지
않아 log-return 이 불연속 점프를 포함한다 → Adj Close 사용이 옳다.

**함정**: `parse_dates=True` 없이 읽으면 인덱스가 문자열 → `.loc[날짜:]` 슬라이싱 실패.

In [2]:
ANALYSIS_START = '2016-01-01'
ANALYSIS_END   = '2025-12-31'

raw_dict = {}
for tic in ['SPY', 'QQQ']:
    df = pd.read_csv(RAW_DATA_DIR / f'{tic}.csv', index_col=0, parse_dates=True)
    df['log_return'] = np.log(df['Adj Close']).diff()  # 누수: 전체 기간에서 diff → 시작일 NaN 방지
    raw_dict[tic] = df.loc[ANALYSIS_START:ANALYSIS_END]

print(f'{"티커":>6}  {"행수":>6}  {"NaN":>4}  {"시작일":>12}  {"종료일":>12}')
print('-' * 55)
for tic, df in raw_dict.items():
    lr = df['log_return']
    print(f'{tic:>6}  {len(df):>6}  {lr.isna().sum():>4}  '
          f'{str(df.index[0].date()):>12}  {str(df.index[-1].date()):>12}')

    티커      행수   NaN           시작일           종료일
-------------------------------------------------------
   SPY    2514     0    2016-01-04    2025-12-31
   QQQ    2514     0    2016-01-04    2025-12-31


## §3. 타깃 생성 — 21일 누적 forward log-return

**무엇을**: Setting A 의 예측 대상인 **21영업일 후 누적 log-return** 을 생성한다.

**왜 21일**: 미국 시장의 평균 영업월(~21 거래일)에 맞춰 월별 리밸런싱 예측을 수행한다.

**공식**:
```
log_ret[t] = log(Adj_Close[t]) - log(Adj_Close[t-1])
target[t]  = Σ log_ret[t+k], k=1..21
           = log_ret.rolling(21).sum().shift(-21)
```

**왜 `.shift(-21)` 인가**:
```
rolling(21).sum() 는 과거 trailing 합 (t-20 ~ t)
.shift(-21)       은 21일 앞으로 당겨서 → 위치 t에 t+1~t+21 합을 배치
```

**shift 부호 그림**:
```
날짜:   ...  t-20  ...  t   t+1  ...  t+21  ...
rolling sum: [t-20~t 합]  ← trailing
.shift(-21): 이 값이 t-21 위치로 이동 → t 위치에는 [t+1~t+21 합]
```

**이것이 누수인가?**: target[t] 자체는 미래값을 포함하지만, 모델이 t 이전 데이터만
입력으로 받으면 누수가 아니다. §6의 Walk-Forward Purge+Embargo가 이를 보장한다.

**NaN 발생 위치**:
- 첫 1행: log_ret.diff() 계산
- 마지막 21행: 미래 21일 데이터 부족

In [3]:
from scripts.targets import build_daily_target_21d

targets_dict = {}
for tic, df in raw_dict.items():
    target = build_daily_target_21d(df['Adj Close'])
    targets_dict[tic] = target
    valid = target.notna().sum()
    print(f'{tic}: 유효 타깃 {valid}개  '
          f'(NaN {target.isna().sum()}개 — 첫 1행 + 마지막 21행 제거 예상)')

print()
print('SPY 타깃 앞 5행 (NaN 포함):')
print(targets_dict['SPY'].head())
print()
print('SPY 타깃 뒤 5행 (NaN 확인):')
print(targets_dict['SPY'].tail())

SPY: 유효 타깃 2493개  (NaN 21개 — 첫 1행 + 마지막 21행 제거 예상)
QQQ: 유효 타깃 2493개  (NaN 21개 — 첫 1행 + 마지막 21행 제거 예상)

SPY 타깃 앞 5행 (NaN 포함):
date
2016-01-04   -0.049561
2016-01-05   -0.049685
2016-01-06   -0.056224
2016-01-07   -0.045492
2016-01-08   -0.034401
Name: Adj Close, dtype: float64

SPY 타깃 뒤 5행 (NaN 확인):
date
2025-12-24   NaN
2025-12-26   NaN
2025-12-29   NaN
2025-12-30   NaN
2025-12-31   NaN
Name: Adj Close, dtype: float64


## §4. 누수 검증 — 3종 체크포인트

**무엇을**: 타깃 시계열이 미래 데이터를 *의도치 않게* 입력에 흘리지 않음을 확인한다.

**왜 3종**: 단일 테스트는 놓칠 수 있는 경우가 있어, López de Prado(2018) 기준으로
단위 테스트 + 육안 확인 + 인공 누수 대조(sanity check) 를 모두 수행한다.

**검증 1**: `assert target[t] == log_ret[t+1:t+22].sum()` — 수치 일치 확인

**검증 2**: 첫 5개 유효 행의 날짜·log_ret·target·직접계산값을 나란히 출력 — 사람이 육안 확인

**검증 3**: 인공 누수 타깃 (`target_leaky[t] = log_ret[t]`) 으로 동일 모델을 1 epoch 학습 →
R² 가 0.9 를 넘으면 평가 파이프라인이 정상. 넘지 않으면 평가 코드에 버그가 있다는 의미.
(검증 3은 §7·§8 모듈 완성 후 활성화)

In [4]:
# 검증 1·2 — assert 단위 테스트 + 육안 확인 표
from scripts.targets import verify_no_leakage

for tic, df in raw_dict.items():
    print(f'\n{'='*60}')
    print(f'  [{tic}] 누수 검증')
    print(f'{'='*60}')
    log_ret = df['log_return'].dropna()          # 분석 기간 내 NaN 없음 (§2에서 확인)
    target  = targets_dict[tic]
    verify_no_leakage(log_ret, target, n_checks=3, seed=42)


  [SPY] 누수 검증
=== 누수 검증 1 — Assert 단위 테스트 ===
  [PASS] 2016-11-17  target=0.035304  직접계산=0.035304  Δ=0.00e+00
  [PASS] 2022-06-27  target=0.031536  직접계산=0.031536  Δ=0.00e+00
  [PASS] 2023-08-31  target=-0.048994  직접계산=-0.048994  Δ=0.00e+00

=== 누수 검증 2 — 육안 확인 표 (첫 5개 유효 행) ===
            날짜     log_ret      target        직접계산    일치
  ------------------------------------------------------
    2016-01-04   -0.014078   -0.049561   -0.049561     O
    2016-01-05    0.001690   -0.049685   -0.049685     O
    2016-01-06   -0.012695   -0.056224   -0.056224     O
    2016-01-07   -0.024284   -0.045492   -0.045492     O
    2016-01-08   -0.011037   -0.034401   -0.034401     O

[OK] 누수 검증 완료 — 모든 체크포인트 PASS

  [QQQ] 누수 검증
=== 누수 검증 1 — Assert 단위 테스트 ===
  [PASS] 2016-11-17  target=0.022546  직접계산=0.022546  Δ=0.00e+00
  [PASS] 2022-06-27  target=0.047935  직접계산=0.047935  Δ=0.00e+00
  [PASS] 2023-08-31  target=-0.043823  직접계산=-0.043823  Δ=0.00e+00

=== 누수 검증 2 — 육안 확인 표 (첫 5개 유효 행) ===
          

In [5]:
# 검증 3 — 인공 누수 대조 (§7·§8 완성 후 활성화)
# from scripts.targets import build_leaky_target_for_test
# leaky_target = build_leaky_target_for_test(raw_dict['SPY']['Adj Close'])
# TODO: 동일 모델 1 epoch 학습 → R² > 0.9 확인
print('[검증 3] 인공 누수 대조 — scripts/models.py 완성 후 활성화 예정')

[검증 3] 인공 누수 대조 — scripts/models.py 완성 후 활성화 예정


## §5. `LSTMDataset` · `make_sequences` — Setting A 파라미터 검증

**무엇을**: `scripts/dataset.py` 의 공개 인터페이스를 로드하고 Setting A 파라미터로 검증한다.

**Setting A `seq_len = 126` (약 6개월 lookback)**: PLAN.md 확정 사양.

**각 샘플 구조**:
```
X[i] = log_return[i : i+126]         # 과거 126일 수익률 시퀀스  shape: (126, 1)
y[i] = target_21d[i+125]             # i+125 일 기준 21일 누적 forward 수익률
```

**폴드별 훈련 샘플 수**:
```
N_train = IS − seq_len = 231 − 126 = 105  (폴드당)
```

**⚠️ (B, T, F) 축 계약 (학습자료_주의사항 §3.3)**:

DataLoader 출력은 `(B, T, F) = (batch, 126, 1)` 형태다.  
PyTorch LSTM의 기본값은 `batch_first=False` → `(T, B, F)` 입력을 기대.  
**`LSTMRegressor` 는 반드시 `batch_first=True` 로 초기화해야 DataLoader 출력과 일치한다.**  
→ `output[:, -1, :]` (마지막 시점 hidden 추출) 도 `batch_first=True` 기준 인덱스임.

| 함수 / 클래스 | 역할 |
|---|---|
| `LSTMDataset` | `(X, y)` 텐서를 보관하는 `torch.utils.data.Dataset` |
| `make_sequences` | 1D/2D 시계열 → 슬라이딩 윈도우 `(X, y)` |
| `walk_forward_folds` | Walk-Forward 폴드 인덱스 목록 생성 |
| `build_fold_datasets` | 폴드 하나의 스케일링 + `LSTMDataset` 일괄 생성 |

In [6]:
from scripts.dataset import (
    LSTMDataset,
    make_sequences,
    walk_forward_folds,
    build_fold_datasets,
)
from torch.utils.data import DataLoader

print('[OK] scripts.dataset import 완료')

[OK] scripts.dataset import 완료


In [7]:
# Setting A 파라미터
SEQ_LEN = 126  # 약 6개월 lookback — PLAN.md 확정값

spy_lr  = raw_dict['SPY']['log_return'].values   # shape (2514,), NaN 없음
qqq_lr  = raw_dict['QQQ']['log_return'].values
spy_tgt = targets_dict['SPY'].values             # shape (2514,), 마지막 21행 NaN

# make_sequences 로 X만 생성해 형태 확인 (y는 target_series 에서 별도 추출)
X_spy, _ = make_sequences(spy_lr, seq_len=SEQ_LEN)

print(f'make_sequences(spy_lr, seq_len={SEQ_LEN}) 결과')
print(f'  X.shape = {X_spy.shape}   → (N, seq_len={SEQ_LEN}, n_features=1)')
print()
print(f'Setting A 폴드별 훈련 샘플 수 (IS=231, seq_len=126):')
print(f'  N_train = IS − seq_len = 231 − 126 = {231 - SEQ_LEN}')
print(f'  N_test  = OOS = 21')
print()
# 정렬 검증: X[0] 마지막 값 = spy_lr[125], target[0] = spy_tgt[125]
print(f'정렬 검증:')
print(f'  X[0][-1] == spy_lr[{SEQ_LEN-1}] → {X_spy[0][-1, 0]:.6f} == {spy_lr[SEQ_LEN-1]:.6f}')
print(f'  target[{SEQ_LEN-1}] (21일 누적 수익률) = {spy_tgt[SEQ_LEN-1]:.6f}')

make_sequences(spy_lr, seq_len=126) 결과
  X.shape = (2388, 126, 1)   → (N, seq_len=126, n_features=1)

Setting A 폴드별 훈련 샘플 수 (IS=231, seq_len=126):
  N_train = IS − seq_len = 231 − 126 = 105
  N_test  = OOS = 21

정렬 검증:
  X[0][-1] == spy_lr[125] → 0.002098 == 0.002098
  target[125] (21일 누적 수익률) = 0.026467


## §6. Walk-Forward fold 생성

**무엇을**: Setting A 의 Walk-Forward 교차검증 폴드 인덱스를 생성하고 검증한다.

**왜 Walk-Forward + Purge + Embargo** (López de Prado 2018 근거):
- **단순 K-Fold 의 문제**: 시계열에서 미래 데이터가 훈련에 포함되어 성능 과대평가
- **Purge(21)**: Setting A 타깃은 21일 forward 합 → IS 끝 21일의 타깃이 OOS 가격을 참조
  → IS 끝 21일을 학습에서 제거해 레이블 겹침 차단
- **Embargo(21)**: IS 직후 구간에 lookback(seq_len=126) autocorrelation 잔존 가능성
  → 추가 21일 공백으로 OOS 성능 부풀림 방지

**폴드 구조**:
```
날짜 축 →
Fold k: ├── IS (231일) ──┤ purge(21) │ emb(21) │ OOS(21) ┤
                                                 ↑ step=21 씩 이동
```

**파라미터**:

| 구성 | 값 | 의미 |
|---|---|---|
| IS | 231 | 훈련 구간 (~11개월) |
| purge | 21 | 레이블 겹침 제거 |
| emb | 21 | autocorrelation 차단 |
| OOS | 21 | 테스트 구간 (~1개월) |
| step | 21 | 폴드 이동 간격 |

**예상 폴드 수**: ⌊(2514 − 294) / 21⌋ = **106**

**IS 내 train/val 재분할** (§8에서 EarlyStopping 용):
- train: IS의 80% = 184일 → 시퀀스 수 = 184 − seq_len = 184 − 126 = 58
- val: IS의 20% = 47일 → 시퀀스 수 = 47 − seq_len = 47 − 126 = **음수!**
  → val 크기 조정 필요: IS 내부 분할 시 `gap = seq_len` 적용 후
    train=IS−seq_len−val_size, val=val_size (val_size는 고정값 사용, 비율 대신)
    (§8 train.py 작성 시 결정)

In [8]:
IS, PURGE, EMB, OOS, STEP = 231, 21, 21, 21, 21
n_samples = len(spy_lr)  # 2514

folds = walk_forward_folds(n_samples, IS, PURGE, EMB, OOS, STEP)
print(f'총 폴드 수: {len(folds)}')
print()

dates = raw_dict['SPY'].index

print(f'{"폴드":>4}  {"train 시작":>12}  {"train 끝":>12}  {"test 시작":>12}  {"test 끝":>12}')
print('-' * 65)
for fold_i in [0, 1, 2, len(folds) - 1]:
    tr, te = folds[fold_i]
    print(f'{fold_i:>4}  {str(dates[tr[0]].date()):>12}  {str(dates[tr[-1]].date()):>12}  '
          f'{str(dates[te[0]].date()):>12}  {str(dates[te[-1]].date()):>12}')

print()
tr0, te0 = folds[0]
print(f'[폴드 0 상세]')
print(f'  train 인덱스: {tr0[0]}~{tr0[-1]}  ({len(tr0)}일)')
print(f'  test  인덱스: {te0[0]}~{te0[-1]}  ({len(te0)}일)')
print(f'  gap (purge+emb): {te0[0] - tr0[-1] - 1}일')
print(f'  훈련 시퀀스 수 (IS − seq_len): {IS - SEQ_LEN}')
print(f'  테스트 입력 lookback 시작: {te0[0] - SEQ_LEN} (purge/emb 구간 포함, 허용)')

총 폴드 수: 106

  폴드      train 시작       train 끝       test 시작        test 끝
-----------------------------------------------------------------
   0    2016-01-04    2016-11-30    2017-02-02    2017-03-03
   1    2016-02-03    2016-12-30    2017-03-06    2017-04-03
   2    2016-03-04    2017-02-01    2017-04-04    2017-05-03
 105    2024-10-08    2025-09-10    2025-11-10    2025-12-09

[폴드 0 상세]
  train 인덱스: 0~230  (231일)
  test  인덱스: 273~293  (21일)
  gap (purge+emb): 42일
  훈련 시퀀스 수 (IS − seq_len): 105
  테스트 입력 lookback 시작: 147 (purge/emb 구간 포함, 허용)


In [9]:
# 폴드 0 — build_fold_datasets 로 텐서 생성 검증 (target_series 사용)
train_idx_0, test_idx_0 = folds[0]

train_ds, test_ds, scaler = build_fold_datasets(
    spy_lr, train_idx_0, test_idx_0, SEQ_LEN,
    target_series=spy_tgt,  # 21일 누적 forward log-return
)

print('=== 폴드 0 텐서 형태 (Setting A: seq_len=126, target=21일 누적) ===')
print(f'train_ds.X : {tuple(train_ds.X.shape)}   (N_train, seq_len, n_feat)')
print(f'train_ds.y : {tuple(train_ds.y.shape)}')
print(f'test_ds.X  : {tuple(test_ds.X.shape)}    (N_test, seq_len, n_feat)')
print(f'test_ds.y  : {tuple(test_ds.y.shape)}')
print()
print('[스케일러 — 훈련 구간 기준 (입력 피처 전용)]')
print(f'  mean  = {scaler.mean_[0]:.8f}   (log_return 평균)')
print(f'  scale = {scaler.scale_[0]:.8f}  (log_return 표준편차)')
print()
print('[타깃 통계 — 비정규화 (21일 누적 수익률)]')
print(f'  train y mean  = {train_ds.y.mean().item():.6f}')
print(f'  train y std   = {train_ds.y.std().item():.6f}')
print(f'  test  y mean  = {test_ds.y.mean().item():.6f}')
print()
print('[텐서 dtype]')
print(f'  X dtype: {train_ds.X.dtype}  (float32 필요)')
print(f'  y dtype: {train_ds.y.dtype}')

=== 폴드 0 텐서 형태 (Setting A: seq_len=126, target=21일 누적) ===
train_ds.X : (105, 126, 1)   (N_train, seq_len, n_feat)
train_ds.y : (105,)
test_ds.X  : (21, 126, 1)    (N_test, seq_len, n_feat)
test_ds.y  : (21,)

[스케일러 — 훈련 구간 기준 (입력 피처 전용)]
  mean  = 0.00040364   (log_return 평균)
  scale = 0.00845726  (log_return 표준편차)

[타깃 통계 — 비정규화 (21일 누적 수익률)]
  train y mean  = 0.010401
  train y std   = 0.021910
  test  y mean  = 0.011888

[텐서 dtype]
  X dtype: torch.float32  (float32 필요)
  y dtype: torch.float32


In [10]:
# DataLoader 배치 검증
# shuffle=True  → 훈련 시 시간 순서 무작위화 (LSTM 학습 관행)
# shuffle=False → 테스트 시 시간 순서 보존
BATCH_SIZE = 32

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=len(test_ds), shuffle=False)

print(f'train_loader: {len(train_loader)} 배치 (batch_size={BATCH_SIZE}, N={len(train_ds)})')
print(f'test_loader : {len(test_loader)} 배치  (전체 한 번에)')
print()

for xb, yb in train_loader:
    print(f'[train 첫 배치] X: {tuple(xb.shape)},  y: {tuple(yb.shape)},  dtype: {xb.dtype}')
    break

for xb, yb in test_loader:
    print(f'[test  전체  ] X: {tuple(xb.shape)},  y: {tuple(yb.shape)}')

train_loader: 4 배치 (batch_size=32, N=105)
test_loader : 1 배치  (전체 한 번에)

[train 첫 배치] X: (32, 126, 1),  y: (32,),  dtype: torch.float32
[test  전체  ] X: (21, 126, 1),  y: (21,)


In [11]:
# 전체 106 폴드 일괄 검증 (SPY)
n_train_list, n_test_list = [], []

for tr_idx, te_idx in folds:
    tr_ds, te_ds, _ = build_fold_datasets(
        spy_lr, tr_idx, te_idx, SEQ_LEN, target_series=spy_tgt
    )
    n_train_list.append(len(tr_ds))
    n_test_list.append(len(te_ds))

print(f'총 {len(folds)} 폴드 정상 생성 완료 (SPY, seq_len={SEQ_LEN})')
print(f'train 샘플 수: 고정={len(set(n_train_list)) == 1},  값={n_train_list[0]}')
print(f'test  샘플 수: 고정={len(set(n_test_list)) == 1},   값={n_test_list[0]}')

총 106 폴드 정상 생성 완료 (SPY, seq_len=126)
train 샘플 수: 고정=True,  값=105
test  샘플 수: 고정=True,   값=21


## §7. LSTMRegressor import — TODO

**필요 파일**: `scripts/models.py`

**구현 예정 내용** (PLAN.md §7):
```python
from scripts.models import LSTMRegressor

# ⚠️ batch_first=True 필수 (학습자료_주의사항 §3.3)
# DataLoader → (B, T, F) = (32, 126, 1)
# LSTM 기본값 batch_first=False → (T, B, F) 기대 → 불일치 발생
# batch_first=True 로 초기화해야 output[:, -1, :] 인덱스가 올바르게 동작
model = LSTMRegressor(
    input_size=1,
    hidden_size=128,
    num_layers=2,
    dropout=0.2,
    batch_first=True,   # (B, T, F) DataLoader 출력과 일치
)
```

**설계 결정 사항**:
- **마지막 시점 hidden 추출** (`output[:, -1, :]`) vs 평균 풀링:
  마지막 시점을 사용한다. LSTM은 순방향으로 과거 정보를 누적하므로
  마지막 hidden state가 전체 시퀀스를 요약한다.
  (`batch_first=True` 기준 — axis 0: batch, axis 1: time, axis 2: hidden)
- 헤드: `Linear(128, 1)` + 활성화 없음 (회귀 출력)

**⚠️ 1-layer dropout 처리** (학습자료_주의사항 §4.4):
PyTorch LSTM의 `dropout` 파라미터는 `num_layers > 1` 일 때만 효과가 있다.
Setting A는 `num_layers=2` 이므로 문제없으나, Setting B (`num_layers=1`) 에서는
별도 `nn.Dropout` 레이어를 Linear 앞에 삽입해야 한다.

In [12]:
# TODO: scripts/models.py 구현 후 활성화
# from scripts.models import LSTMRegressor
# model = LSTMRegressor(input_size=1, hidden_size=128, num_layers=2, dropout=0.2)
# print(model)
print('[§7] scripts/models.py 미구현 — 다음 단계에서 작성 예정')

[§7] scripts/models.py 미구현 — 다음 단계에서 작성 예정


## §8. 학습 루프 — TODO

**필요 파일**: `scripts/train.py`

**구현 예정 내용** (PLAN.md §8):
```python
from scripts.train import train_one_fold

result = train_one_fold(model, train_loader, val_loader, **hp)
# result = {'best_state_dict': ..., 'history': {'train_loss': [], 'val_loss': []}}
```

**하이퍼파라미터 (PLAN.md 확정)**:

| 항목 | 값 |
|---|---|
| Loss | Huber(delta=0.01) |
| Optimizer | AdamW(lr=1e-3, weight_decay=1e-4) |
| Scheduler | ReduceLROnPlateau(patience=5, factor=0.5) |
| Gradient clip | max_norm=1.0 |
| EarlyStopping | patience=10 |
| Checkpoint | best val loss 기준 |

**val 크기 결정**: IS=231, seq_len=126 → 훈련 가능 시퀀스 105개
80/20 분할 시 val=21개 (고정), train=84개. gap = seq_len 은 IS 내부 고려.

**함정**: `optimizer.zero_grad()` 를 `loss.backward()` 전에 반드시 호출.

In [13]:
# TODO: scripts/train.py 구현 후 활성화
# from scripts.train import train_one_fold
# ...
print('[§8] scripts/train.py 미구현 — 다음 단계에서 작성 예정')

[§8] scripts/train.py 미구현 — 다음 단계에서 작성 예정


## §9. 평가·시각화 — TODO

**필요 파일**: `scripts/metrics.py`, `scripts/plot_utils.py`

**구현 예정 내용** (PLAN.md §9):
```python
from scripts.metrics import hit_rate, r2_oos, baseline_metrics
from scripts.plot_utils import plot_learning_curve, plot_pred_vs_actual
```

**1차 지표 (관문 판정)**:

| 지표 | 공식 | 관문 |
|---|---|---|
| Hit Rate | `mean(sign(y_pred) == sign(y_true))` | > 0.55 |
| R²_OOS | `1 − SSE_model / sum(y²)` (Campbell & Thompson 2008) | > 0 |

**R²_OOS 해석**: 0보다 크면 '0 예측 베이스라인' 대비 개선. 음수이면 모델이
아무것도 예측하지 않는 것보다 못하다는 의미 — 즉시 모델 재검토.

**결과 저장**: `results/setting_A/{ticker}/metrics.json`

In [14]:
# TODO: scripts/metrics.py, scripts/plot_utils.py 구현 후 활성화
# from scripts.metrics import hit_rate, r2_oos, baseline_metrics
# from scripts.plot_utils import plot_learning_curve, plot_pred_vs_actual
# ...
print('[§9] scripts/metrics.py · plot_utils.py 미구현 — 다음 단계에서 작성 예정')

[§9] scripts/metrics.py · plot_utils.py 미구현 — 다음 단계에서 작성 예정


## §10. 결론·체크포인트

### §1~§6 완료 사항 요약

| 항목 | 결과 |
|---|---|
| 분석 기간 | 2016-01-04 ~ 2025-12-31, SPY·QQQ 각 2514행 |
| log_return NaN | 0 (분석 기간 내) |
| 타깃 유효 행 | 2493개 (NaN 21개: 마지막 21행) |
| 누수 검증 | SPY·QQQ 모두 Assert PASS + 육안 표 확인 |
| seq_len | **126** (Setting A 확정) |
| 폴드 수 | **106** (SPY·QQQ 동일) |
| 훈련 샘플/폴드 | **105** (IS 231 − seq_len 126) |
| 테스트 샘플/폴드 | **21** (OOS 21) |
| (B,T,F) 축 | DataLoader: `(32,126,1)` → `batch_first=True` 필수 확인 |

### 학습자료_주의사항 준수 현황

| 항목 | 상태 |
|---|---|
| 1.3 Normalization Leakage | ✅ `scaler.fit(train_idx)` only |
| 1.4 Window Leakage | ✅ purge+embargo 시간 기반 분리 |
| 1.5 Walk-Forward purge+embargo | ✅ purge=21, emb=21 적용 |
| 2.2 log-return 사용 | ✅ `np.log(Adj Close).diff()` |
| 3.3 (B,T,F) 축 계약 | ✅ 명시됨 — `batch_first=True` 요구사항 §5·§7에 기록 |
| 4.4 1-layer dropout 함정 | ✅ §7에 주의사항 기록 |
| 7.1 fit/transform 분리 | ✅ `build_fold_datasets` 내 분리 |

### 다음 단계 (§7~§9)

1. `scripts/models.py` — `LSTMRegressor(batch_first=True)` 구현
2. `scripts/train.py` — `train_one_fold` 구현 (Huber, AdamW, EarlyStopping)
3. `scripts/metrics.py` — `hit_rate`, `r2_oos`, `baseline_metrics` 구현
4. `scripts/plot_utils.py` — 한글 폰트 + 표준 플롯 함수
5. §4 검증 3 (인공 누수 대조) 활성화
6. 전체 106 폴드 학습 실행 및 SPY·QQQ 각각 `metrics.json` 저장